# Chapter 9
Use the `qiskit` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=48, noise=0.20, random_state=7)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
# Reuse these angle arrays for every quantum evaluation.
X_train = np.clip(X_train_scaled, -np.pi, np.pi)
X_test = np.clip(X_test_scaled, -np.pi, np.pi)

K_train_linear = X_train_scaled @ X_train_scaled.T
K_test_linear = X_test_scaled @ X_train_scaled.T
linear_svc = SVC(kernel="precomputed", C=1.0)
linear_svc.fit(K_train_linear, y_train)
linear_pred = linear_svc.predict(K_test_linear)
print("Train/test shapes:", X_train.shape, X_test.shape)
print("Linear balanced accuracy:",
      balanced_accuracy_score(y_test, linear_pred))


In [ ]:
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.primitives import QMLSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute

feature_map = zz_feature_map(
    feature_dimension=2, reps=2, entanglement="linear"
)

def make_kernel(shots, seed=19):
    """Return an uncorrected fidelity kernel; None selects exact probabilities."""
    sampler = QMLSampler(shots=shots, seed=np.random.default_rng(seed))
    return FidelityQuantumKernel(
        feature_map=feature_map,
        fidelity=ComputeUncompute(sampler=sampler, local=False),
        enforce_psd=False,
        evaluate_duplicates="none",
    )

exact_kernel = make_kernel(shots=None)
K_train_exact = exact_kernel.evaluate(X_train)
K_test_exact = exact_kernel.evaluate(X_test, X_train)

shots = 512
shot_kernel = make_kernel(shots=shots)
K_train_raw = shot_kernel.evaluate(X_train)
K_test_raw = shot_kernel.evaluate(X_test, X_train)

print("Gram shapes:", K_train_exact.shape, K_test_exact.shape)
print("Exact training block:\n", np.round(K_train_exact[:3, :3], 3))
print("Shot training block:\n", np.round(K_train_raw[:3, :3], 3))
print("Shot train RMSE:",
      np.sqrt(np.mean((K_train_raw - K_train_exact) ** 2)))


In [ ]:
# Fit the correction on training similarities only.
K_symmetric = (K_train_raw + K_train_raw.T) / 2
eigenvalues, eigenvectors = np.linalg.eigh(K_symmetric)
keep = eigenvalues > 1e-10
V = eigenvectors[:, keep]
positive_values = eigenvalues[keep]
K_train_quantum = (V * positive_values) @ V.T
# Extend the retained training subspace to test-to-training entries.
K_test_quantum = (K_test_raw @ V) @ V.T

print("Minimum raw/corrected eigenvalues:",
      eigenvalues.min(), np.linalg.eigvalsh(K_train_quantum).min())
print("Relative training correction:",
      np.linalg.norm(K_train_quantum - K_train_raw)
      / np.linalg.norm(K_train_raw))

kernel_pairs = {
    "exact": (K_train_exact, K_test_exact),
    "shots raw": (K_train_raw, K_test_raw),
    "shots PSD": (K_train_quantum, K_test_quantum),
}
quantum_models = {}
quantum_predictions = {}
for name, (K_fit, K_predict) in kernel_pairs.items():
    model = SVC(kernel="precomputed", C=1.0).fit(K_fit, y_train)
    prediction = model.predict(K_predict)
    quantum_models[name] = model
    quantum_predictions[name] = prediction
    print(name, "balanced accuracy:",
          balanced_accuracy_score(y_test, prediction))


In [ ]:
def kernel_diagnostics(K, y):
    """Summarize a square Gram matrix for two classes with repeated labels."""
    K = 0.5 * (K + K.T)
    y_pm = np.where(y > 0, 1.0, -1.0)

    same = y_pm[:, None] == y_pm[None, :]
    off_diagonal = ~np.eye(len(y), dtype=bool)

    same_mean = K[same & off_diagonal].mean()
    diff_mean = K[(~same) & off_diagonal].mean()

    H = np.eye(len(y)) - np.ones_like(K) / len(y)
    K_centered = H @ K @ H
    Y_centered = H @ np.outer(y_pm, y_pm) @ H
    denominator = np.linalg.norm(K_centered) * np.linalg.norm(Y_centered)
    alignment = (np.sum(K_centered * Y_centered) / denominator
                 if denominator > 1e-12 else np.nan)

    eigenvalues = np.linalg.eigvalsh(K)
    return {
        "same_mean": same_mean,
        "diff_mean": diff_mean,
        "similarity_gap": same_mean - diff_mean,
        "alignment": alignment,
        "min_eigenvalue": eigenvalues.min(),
        "max_eigenvalue": eigenvalues.max(),
    }

for name, (K_fit, _) in kernel_pairs.items():
    report = kernel_diagnostics(K_fit, y_train)
    print(name, {key: round(float(value), 4)
                 for key, value in report.items()})


In [ ]:
# Continue with the same split, scaler, labels, and quantum matrices.
classical_models = {
    "linear": SVC(kernel="linear", C=1.0),
    "rbf": SVC(kernel="rbf", C=1.0, gamma="scale"),
    "poly2": SVC(kernel="poly", degree=2, C=1.0),
}
for name, model in classical_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    print(name, balanced_accuracy_score(y_test, y_pred))

def trig_features(A):
    """Return sine/cosine features of local and ZZ interaction phases."""
    pair_phase = ((np.pi - A[:, 0]) * (np.pi - A[:, 1]))[:, None]
    phases = 2 * np.hstack([A, pair_phase])
    return np.hstack([np.sin(phases), np.cos(phases)])

trig_svc = SVC(kernel="rbf", C=1.0, gamma="scale")
trig_svc.fit(trig_features(X_train), y_train)
trig_pred = trig_svc.predict(trig_features(X_test))
print("trigonometric", balanced_accuracy_score(y_test, trig_pred))

# Reuse predictions; do not re-estimate the shot matrices.
for name, prediction in quantum_predictions.items():
    print("quantum " + name, balanced_accuracy_score(y_test, prediction))
